# Exploração para anteprojeto: sífilis congênita em Porto Alegre

Este notebook reaproveita a leitura dos microdados do laboratório, mas tem outro objetivo: testar perguntas analíticas para o anteprojeto sobre desigualdade racial, pré-natal e perfil socioeconômico.

O notebook do laboratório permanece sendo `visualizacao_sifilis_congenita_poars.ipynb`. Aqui podemos comparar perguntas alternativas sem poluir a entrega principal.


## 1. Dependências

O Google Colab já inclui `pandas`, `matplotlib`, `seaborn` e `numpy`. Para este notebook, as únicas dependências adicionais são `pyreaddbc`, usada para converter arquivos `.dbc` do DATASUS para `.dbf`, e `dbfread`, usada para ler o `.dbf` convertido.

A célula abaixo só instala esses pacotes se eles ainda não estiverem disponíveis. Ela não reinstala bibliotecas nativas do Colab e não reinicia o runtime automaticamente.


In [ ]:
import importlib.util
import subprocess
import sys

pacotes = []
if importlib.util.find_spec("pyreaddbc") is None:
    pacotes.append("pyreaddbc==2.0.4")
if importlib.util.find_spec("dbfread") is None:
    pacotes.append("dbfread==2.0.7")

if pacotes:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", *pacotes])
    print("Pacotes instalados:", ", ".join(pacotes))
else:
    print("Dependências já instaladas.")


## 2. Bibliotecas e configurações

Esta seção usa as bibliotecas já disponíveis no Colab para manipulação e visualização dos dados.


In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", context="notebook")

CODIGO_PORTO_ALEGRE = "431490"


## 3. Caminhos dos dados

A estrutura esperada no repositório é:

- `data/raw/SIFCBR24.dbc`
- `data/raw/sinasc/DNRS2024.dbc`
- `data/raw/cnes/STRS24*.dbc`


In [ ]:
def encontrar_arquivo(*candidatos):
    for candidato in candidatos:
        caminho = Path(candidato)
        if caminho.exists():
            return caminho
    caminhos = ", ".join(str(c) for c in candidatos)
    raise FileNotFoundError(f"Nenhum arquivo encontrado. Caminhos testados: {caminhos}")

ARQUIVO_SINAN = encontrar_arquivo(
    "../data/raw/SIFCBR24.dbc",
    "data/raw/SIFCBR24.dbc",
    "SIFCBR24.dbc",
)

ARQUIVO_SINASC = encontrar_arquivo(
    "../data/raw/sinasc/DNRS2024.dbc",
    "data/raw/sinasc/DNRS2024.dbc",
    "DNRS2024.dbc",
)

print("SINAN:", ARQUIVO_SINAN)
print("SINASC:", ARQUIVO_SINASC)


## 4. Leitura dos arquivos `.dbc`

O pacote `pyreaddbc` converte os arquivos comprimidos `.dbc` do DATASUS para `.dbf`. Em seguida, `dbfread` lê o `.dbf` convertido e o resultado é transformado em um DataFrame pandas.


In [ ]:
import tempfile
from pathlib import Path

import pyreaddbc
from dbfread import DBF, FieldParser


class ParserDatasInvalidas(FieldParser):
    def parseD(self, field, data):
        try:
            return super().parseD(field, data)
        except ValueError:
            return None


def ler_dbc(caminho_dbc, encoding="iso-8859-1"):
    caminho_dbc = Path(caminho_dbc)
    with tempfile.TemporaryDirectory() as tmpdir:
        caminho_dbf = Path(tmpdir) / f"{caminho_dbc.stem}.dbf"
        pyreaddbc.dbc2dbf(str(caminho_dbc), str(caminho_dbf))
        tabela = DBF(
            str(caminho_dbf),
            encoding=encoding,
            parserclass=ParserDatasInvalidas,
            load=True,
        )
        return pd.DataFrame(iter(tabela))


sinan = ler_dbc(ARQUIVO_SINAN)
sinasc = ler_dbc(ARQUIVO_SINASC)

print("SINAN", sinan.shape)
print("SINASC", sinasc.shape)


## 5. Recorte: Porto Alegre

Nos microdados, o código de Porto Alegre aparece como `431490` ou com dígito adicional. Por isso o filtro usa `startswith("431490")`.


In [ ]:
sinan_poa = sinan[
    sinan["ID_MN_RESI"].astype(str).str.strip().str.startswith(CODIGO_PORTO_ALEGRE)
].copy()

sinasc_poa = sinasc[
    sinasc["CODMUNRES"].astype(str).str.strip().str.startswith(CODIGO_PORTO_ALEGRE)
].copy()

print("Casos SINAN Porto Alegre:", len(sinan_poa))
print("Nascidos vivos SINASC Porto Alegre:", len(sinasc_poa))


## 6. Perguntas explorat?rias

As perguntas abaixo seguem o foco do anteprojeto: entender se raça/cor, acesso ao pré-natal e escolaridade materna aparecem combinados nos casos de sífilis congênita em Porto Alegre.

Perguntas iniciais:

1. Dentro dos casos sem pré-natal, qual é a escolaridade por grupo racial?
2. Considerando todos os casos de sífilis congênita, como pré-natal e escolaridade se combinam por grupo racial?
3. A baixa escolaridade é mais frequente entre casos sem pré-natal do que entre casos com pré-natal?


In [ ]:
def grupo_raca_comparativo(valor):
    codigo = str(valor).strip()
    if codigo in {"2", "4"}:
        return "Mães negras"
    if codigo in {"1", "3", "5"}:
        return "Mães não negras"
    return "Ignorado/sem informação"


def grupo_escolaridade_sinan(valor):
    codigo = str(valor).strip()
    if codigo in {"02", "03", "04", "05"}:
        return "Até 7 anos"
    if codigo in {"06", "07", "08"}:
        return "8 anos ou mais"
    return "Ignorada/sem informação"


def grupo_prenatal_sinan(valor):
    codigo = str(valor).strip()
    if codigo == "1":
        return "Com pré-natal"
    if codigo == "2":
        return "Sem pré-natal"
    return "Ignorado/sem informação"


ordem_raca = ["Mães não negras", "Mães negras"]
ordem_escolaridade = ["Até 7 anos", "8 anos ou mais", "Ignorada/sem informação"]
ordem_prenatal = ["Com pré-natal", "Sem pré-natal", "Ignorado/sem informação"]

casos = sinan_poa.copy()
casos["grupo_raca"] = casos["ANT_RACA"].map(grupo_raca_comparativo)
casos["grupo_escolaridade"] = casos["ESCOLMAE"].map(grupo_escolaridade_sinan)
casos["grupo_prenatal"] = casos["ANT_PRE_NA"].map(grupo_prenatal_sinan)
casos = casos[casos["grupo_raca"].isin(ordem_raca)].copy()

print("Casos analisados:", len(casos))
display(casos[["grupo_raca", "grupo_prenatal", "grupo_escolaridade"]].value_counts().reset_index(name="casos"))


## 7. Pergunta 1: dentro dos casos sem pré-natal, qual é a escolaridade por grupo racial?

Esta é a pergunta usada no notebook do laboratório. Ela é precisa, mas trabalha com um subconjunto pequeno dos casos.


In [ ]:
sem_prenatal = casos[casos["grupo_prenatal"].eq("Sem pré-natal")].copy()

q1 = (
    sem_prenatal
    .groupby(["grupo_raca", "grupo_escolaridade"], as_index=False)
    .size()
    .rename(columns={"size": "casos"})
)
q1["total_grupo"] = q1.groupby("grupo_raca")["casos"].transform("sum")
q1["percentual"] = q1["casos"] / q1["total_grupo"] * 100
q1.sort_values(["grupo_raca", "grupo_escolaridade"])


## 8. Pergunta 2: em todos os casos, como pré-natal e escolaridade se combinam por grupo racial?

Esta pergunta usa mais casos e ajuda a ver a combinação entre acesso ao pré-natal e escolaridade sem restringir a análise apenas aos casos sem pré-natal.


In [ ]:
casos["combinacao_acesso"] = casos["grupo_prenatal"] + " | " + casos["grupo_escolaridade"]

q2 = (
    casos
    .groupby(["grupo_raca", "grupo_prenatal", "grupo_escolaridade"], as_index=False)
    .size()
    .rename(columns={"size": "casos"})
)
q2["total_grupo"] = q2.groupby("grupo_raca")["casos"].transform("sum")
q2["percentual"] = q2["casos"] / q2["total_grupo"] * 100
q2.sort_values(["grupo_raca", "grupo_prenatal", "grupo_escolaridade"])


## 9. Pergunta 3: baixa escolaridade é mais frequente quando não houve pré-natal?

Aqui a comparação desloca o foco: em vez de olhar apenas raça/cor, ela pergunta se a ausência de pré-natal se associa à baixa escolaridade dentro de cada grupo racial.


In [ ]:
casos_validos_prenatal = casos[casos["grupo_prenatal"].isin(["Com pré-natal", "Sem pré-natal"])].copy()
casos_validos_prenatal["baixa_escolaridade"] = casos_validos_prenatal["grupo_escolaridade"].eq("Até 7 anos")

q3 = (
    casos_validos_prenatal
    .groupby(["grupo_raca", "grupo_prenatal"], as_index=False)
    .agg(
        baixa_escolaridade=("baixa_escolaridade", "sum"),
        total=("baixa_escolaridade", "size"),
    )
)
q3["percentual_baixa_escolaridade"] = q3["baixa_escolaridade"] / q3["total"] * 100
q3.sort_values(["grupo_raca", "grupo_prenatal"])
